# Interactive Plotting with Matplotlib Widgets

This example demonstrates how to create interactive plots using Matplotlib's widget system. We create a simple sine curve with a movable point controlled by a slider.

## Setup

For interactive widgets to work in Jupyter notebooks, we need to use the appropriate backend. The `%matplotlib widget` magic enables interactive features.

In [ ]:
# For interactivity in Jupyter notebooks, use one of these backends:
# Option 1: ipympl (recommended) - install with: pip install ipympl
# %matplotlib widget

# Option 2: notebook backend (older) - install with: pip install ipywidgets
# %matplotlib notebook

# Option 3: For non-interactive use, comment out the magic command above

# If you get "RuntimeError: 'widget' is not a recognised backend name":
# 1. Install ipympl: pip install ipympl
# 2. Restart your Jupyter kernel
# 3. Run: %matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

## Basic Interactive Plot with Slider

The workflow for creating an interactive plot involves four main steps:

1. **Create the base plot** with the curve and initial point
2. **Create a slider widget** positioned below the main plot
3. **Define an update function** that changes the plot when the slider moves
4. **Connect the slider to the update function** using `on_changed()`

Let us implement this step by step.

In [ ]:
# Step 1: Create the base plot
# Generate sine curve data
x = np.linspace(0, 2*np.pi, 200)
y = np.sin(x)

# Create figure with extra space at bottom for slider
fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(bottom=0.25)

# Plot the sine curve
line, = ax.plot(x, y, 'b-', linewidth=2, label='sin(x)')
ax.set_xlabel('x [rad]')
ax.set_ylabel('y')
ax.set_title('Interactive Sine Curve')
ax.grid(True, alpha=0.3)
ax.legend()

# Create initial point at x = 0
x_initial = 0
y_initial = np.sin(x_initial)
dot, = ax.plot(x_initial, y_initial, 'ro', markersize=12)

# Add text to display current coordinates
coord_text = ax.text(0.02, 0.95, '', transform=ax.transAxes,
                     fontsize=10, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Step 2: Create slider widget
# Position: [left, bottom, width, height] in figure coordinates (0 to 1)
slider_ax = plt.axes([0.15, 0.1, 0.7, 0.03])
slider = Slider(
    ax=slider_ax,
    label='x position',
    valmin=0,
    valmax=2*np.pi,
    valinit=x_initial,
    color='lightblue'
)

# Step 3: Define update function
def update(val):
    """Update dot position when slider moves."""
    # Get current slider value
    x_pos = slider.val
    y_pos = np.sin(x_pos)
    
    # Update dot position
    dot.set_data([x_pos], [y_pos])
    
    # Update coordinate display
    coord_text.set_text(f'x = {x_pos:.3f} rad\ny = sin(x) = {y_pos:.3f}')
    
    # Redraw canvas (works with all backends)
    fig.canvas.draw()

# Step 4: Connect slider to update function
slider.on_changed(update)

# Initialize display
update(x_initial)

plt.show()

## How It Works

### Creating Space for the Slider

The `plt.subplots_adjust(bottom=0.25)` command reserves space at the bottom of the figure for the slider widget. Without this adjustment, the slider would overlap with the x-axis labels.

### Slider Positioning

The slider position is specified using figure coordinates, where (0, 0) is the bottom-left corner and (1, 1) is the top-right corner. The format is `[left, bottom, width, height]`. For our slider at `[0.15, 0.1, 0.7, 0.03]`:

- Starts 15% from the left edge
- Positioned 10% from the bottom
- Spans 70% of the figure width
- Has a height of 3% of the figure height

### The Update Function

The update function is called automatically whenever the slider value changes. It receives the new value as an argument (though we access it via `slider.val` for clarity). The function must update any plot elements that depend on the slider value and then redraw the canvas using `fig.canvas.draw()`, which works across all Matplotlib backends.

### Event Connection

The `slider.on_changed(update)` call registers our update function as a callback. Matplotlib's event system ensures that whenever the user moves the slider, our function executes automatically.

## Multiple Sliders Example

We can extend this pattern to control multiple parameters. Here we add a second slider to control the amplitude of the sine wave.

In [ ]:
# Create figure with extra space for two sliders
fig2, ax2 = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(bottom=0.3)

# Generate initial curve
x = np.linspace(0, 2*np.pi, 200)
amplitude_init = 1.0
x_pos_init = np.pi / 2
y = amplitude_init * np.sin(x)

# Plot curve and point
line2, = ax2.plot(x, y, 'b-', linewidth=2, label='A·sin(x)')
dot2, = ax2.plot(x_pos_init, amplitude_init * np.sin(x_pos_init), 
                 'ro', markersize=12)

ax2.set_xlabel('x [rad]')
ax2.set_ylabel('y')
ax2.set_title('Interactive Sine with Amplitude Control')
ax2.set_ylim(-3, 3)
ax2.grid(True, alpha=0.3)
ax2.legend()

# Add coordinate text
coord_text2 = ax2.text(0.02, 0.95, '', transform=ax2.transAxes,
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Create position slider
slider_pos_ax = plt.axes([0.15, 0.15, 0.7, 0.03])
slider_pos = Slider(slider_pos_ax, 'x position', 0, 2*np.pi, 
                    valinit=x_pos_init, color='lightblue')

# Create amplitude slider
slider_amp_ax = plt.axes([0.15, 0.08, 0.7, 0.03])
slider_amp = Slider(slider_amp_ax, 'Amplitude', 0.1, 3.0, 
                    valinit=amplitude_init, color='lightcoral')

# Update function that responds to both sliders
def update_both(val):
    """Update curve and dot when either slider moves."""
    amp = slider_amp.val
    x_pos = slider_pos.val
    
    # Update the curve
    y_new = amp * np.sin(x)
    line2.set_ydata(y_new)
    
    # Update the dot
    y_pos = amp * np.sin(x_pos)
    dot2.set_data([x_pos], [y_pos])
    
    # Update text
    coord_text2.set_text(
        f'Amplitude = {amp:.2f}\nx = {x_pos:.3f} rad\ny = {y_pos:.3f}'
    )
    
    # Redraw canvas (works with all backends)
    fig2.canvas.draw()

# Connect both sliders to the same update function
slider_pos.on_changed(update_both)
slider_amp.on_changed(update_both)

# Initialize
update_both(None)

plt.show()

## Key Principles for Interactive Plots

**Keep update functions fast**: The update function executes on every slider movement, so it must be efficient. Avoid expensive computations when possible. If calculations are necessary, consider computing them once and storing results rather than recalculating on every update.

**Update only what changes**: In the two-slider example, we update both the curve's y-data and the dot's position because both depend on the slider values. If only the dot position changed, we would update only the dot to maintain responsiveness.

**Use appropriate backends**: In Jupyter notebooks, use `%matplotlib widget` for interactive widgets. For standalone scripts, the default backend usually works, but `%matplotlib qt5` can provide better performance for complex interactions.

**Design for clarity**: Interactive plots should make relationships between variables immediately apparent. In our examples, moving the position slider shows how the point traces the curve, while the amplitude slider demonstrates how the function scales.

## Other Widget Options

Matplotlib provides several other interactive widgets beyond sliders:

- `Button`: Execute an action when clicked
- `CheckButtons`: Toggle multiple options
- `RadioButtons`: Select one option from several
- `TextBox`: Enter numerical or text values
- `RangeSlider`: Select a range rather than a single value

These widgets follow the same pattern: create the widget, define an update function, and connect them using `on_clicked()`, `on_changed()`, or the appropriate callback method.